# Lab 2 - An A/B Test, End to End

*SDAIA Academy · Experimentation and Causal Inference · STARTER notebook*

## Objective
Plan an experiment with a real power calculation, verify the pipeline with an A/A test and a sample-ratio check, estimate the effect two ways (simple and CUPED-adjusted), check guardrails and segments with the right multiple-testing family, and show why peeking is not benign and how an always-valid boundary repairs it.

Dataset: `injaz_experiment_results.csv` - the randomized **guided uploader** test (80,000 users, hash-assigned 50/50). Columns: `user_id, variant, completed, pre_completion_rate, time_to_complete, support_ticket, upload_error, region, device, age, digital_literacy`.

In [ ]:
import sys; sys.path.insert(0, '..')   # so `causal_utils` is importable from starter/ or solution/
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf
from causal_utils import *
plt.rcParams['figure.figsize'] = (7, 3.5)
rng = np.random.default_rng(213)
DATA = '../data'

In [ ]:
df = pd.read_csv(f'{DATA}/injaz_experiment_results.csv')
print(df.shape); df.head()

## Step 1 - Power, MDE and sample size
Baseline completion 0.55, MDE = +2 percentage points, alpha 0.05 two-sided, power 0.80. Cross-check the closed form against `statsmodels`. Then plot n vs MDE and produce the run-length plan (12,000 eligible users/day, trigger rate 0.6, 14-day floor).

In [ ]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize
p1, mde = 0.55, 0.02
n_closed = sample_size_proportions(p1, mde)              # closed form (causal_utils)
n_sm = ...   # TODO: NormalIndPower().solve_power(effect_size=proportion_effectsize(p1+mde, p1), alpha=0.05, power=0.8)
print(f"n per arm: closed-form {n_closed:,}   statsmodels {int(np.ceil(n_sm)):,}")

# TODO: plot n per arm vs MDE in [0.005, 0.05] (log y axis)
# TODO: MDE achievable with 12,000 eligible/day * 0.6 trigger * 28 days split over 2 arms  -> mde_proportions(...)
# TODO: run_length_days(n_closed, arms, 12_000, 0.6) for 2 and 3 arms

## Step 2 - Integrity: reproduce the assignment, run the SRM check and the balance table
Re-derive the arm from the user_id with `assign_variant(uid, 'injaz-uploader-2026q2')`; it must match 100%. Then break the data on purpose (drop 3% of treatment rows) and confirm SRM flags it.

In [ ]:
rederived = df['user_id'].map(lambda u: assign_variant(u, 'injaz-uploader-2026q2'))
print("Assignment reproducible:", (rederived == df['variant']).mean())
counts = df['variant'].value_counts().to_dict()
print("SRM clean :", srm_check(counts, {'control': .5, 'treatment': .5}))
# TODO: drop 3% of treatment rows and re-run srm_check -> it must say INVALID
df['T'] = (df['variant'] == 'treatment').astype(int)
# TODO: balance_table(df, ['age', 'digital_literacy', 'pre_completion_rate'], 'T')

## Step 3 - A/A test on simulated traffic
Simulate a data-generating process with a pre-period covariate correlated with the outcome, assign two arms with **no** effect, and confirm the pipeline returns a null result and a clean ratio. Repeat 500 times: the false-positive rate must be about 5%.

In [ ]:
def simulate_traffic(n=20_000, effect=0.0, seed=0):
    r = np.random.default_rng(seed)
    pre = np.clip(r.normal(0.55, 0.15, n), 0, 1)
    t = r.binomial(1, 0.5, n)
    p = np.clip(0.12 + 0.75 * pre + effect * t, 0.01, 0.99)
    return pd.DataFrame({'T': t, 'pre': pre, 'y': r.binomial(1, p)})

aa = simulate_traffic(seed=42)
print("A/A estimate:", diff_in_means(aa['y'].values, aa['T'].values))
# TODO: loop 500 seeds, collect p_value < 0.05, print the false-positive rate (expect ~0.05)

## Step 4 - Estimate the uploader effect two ways: simple difference vs CUPED-adjusted
Run (a) a two-proportion test, (b) OLS with HC3 robust SEs, (c) OLS with `pre_completion_rate` as a covariate, and (d) explicit CUPED on the outcome. Compare the standard errors and the variance reduction.

In [ ]:
from statsmodels.stats.proportion import proportions_ztest
simple = diff_in_means(df['completed'].values, df['T'].values)
print("Simple difference:", simple)
m_robust = smf.ols('completed ~ T', data=df).fit(cov_type='HC3')
m_adj = ...        # TODO: add pre_completion_rate as covariate, HC3
# TODO: print lift, SE and CI for both models (m.params['T'], m.bse['T'], m.conf_int().loc['T'])
df['completed_cuped'] = cuped_adjust(df['completed'], df['pre_completion_rate'])
cu = diff_in_means(df['completed_cuped'].values, df['T'].values)
vr = ...           # TODO: 1 - var(cuped)/var(raw)
print(f"CUPED lift={cu['estimate']:+.4f} SE={cu['se']:.4f} variance reduced {vr:.1%}")

## Step 5 - Guardrails (Bonferroni) and a segment sweep (Benjamini-Hochberg)
Guardrails: `time_to_complete` (log scale), `support_ticket`, `upload_error`. A guardrail family uses FWER control. The exploratory region x device sweep uses FDR control; survivors are hypotheses for the next test, not findings.

In [ ]:
from statsmodels.stats.multitest import multipletests
df['log_ttc'] = np.log(df['time_to_complete'])
# TODO: for each guardrail fit smf.ols(f'{g} ~ T', df).fit(cov_type='HC3'); collect p-values; multipletests(..., method='bonferroni')
# TODO: segment sweep over region x device: diff_in_means per segment, then multipletests(method='fdr_bh'); count raw vs BH survivors

## Step 6 - Peeking: 1,000 experiments with daily looks under the null
Stop at the first p < 0.05. Measure the realised false-positive rate for 1, 5, 10 and 20 looks.

In [ ]:
def peeking_fpr(n_looks, per_look=500, n_sims=1000, alpha=0.05, seed=4):
    r = np.random.default_rng(seed); fp = 0
    for _ in range(n_sims):
        a = b = np.empty(0); hit = False
        for _ in range(n_looks):
            # TODO: append per_look draws from N(0,1) to BOTH arms (no true effect), t-test, break if p < alpha
            ...
        fp += hit
    return fp / n_sims

for looks in (1, 5, 10, 20):
    print(f"{looks:2d} looks -> false-positive rate {peeking_fpr(looks):.1%}")

## Step 7 - Repeat under an always-valid boundary (mSPRT)
`msprt_reject(diff, var_diff)` returns True when the mixture likelihood ratio crosses 1/alpha. It may be checked at every look. The error rate should return to (below) nominal, and it still lets you stop early when a real effect exists.

In [ ]:
# TODO: copy the peeking loop, but replace the t-test with msprt_reject(diff, var, tau2=0.05**2)
#       where diff = a.mean()-b.mean() and var = a.var(ddof=1)/len(a) + b.var(ddof=1)/len(b)
# Report the rejection rate at 20 looks with effect 0 (should be <= 5%) and with effect 0.1 (power)

## Step 8 - Decision paragraph
Write four sentences: the lift and its CI, the guardrail verdict, whether the lift is *practically* significant against the 2-point MDE, and the one-line peeking rule this lab justifies.

*Your answer:*

...

### Fast finishers
Analyse `completed` at a fake session level (duplicate each user 3 times with correlated noise) with and without `cov_type='cluster'`. Watch the naive CI become dishonestly narrow.